In [20]:
from datetime import datetime
from pymongo import MongoClient
import psycopg2
from psycopg2.extras import execute_batch

In [21]:
#!pip install mysql-connector-python

In [ ]:
# CONFIG 
MONGO_URI = "mongodb://localhost:27017"
MONGO_DB = "ny_project"

PG_HOST = "localhost"
PG_PORT = 5432
PG_DB = "energy_analysis_db"
PG_USER = "postgres"
PG_PASSWORD = "1234"  # change

# Mongo collections
COL_ACS = "acs_zip_population"


In [23]:
# MongoDB connection
mongo = MongoClient(MONGO_URI)[MONGO_DB]

# PostgreSQL connection
pg = psycopg2.connect(
    host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PASSWORD
)
pg.autocommit = True

In [24]:
#data migration
def migrate_acs():
    """
    Migrate ACS population data from MongoDB to PostgreSQL.
    
    Reads data from MongoDB collection and inserts into PostgreSQL table
    using batch processing for performance. Handles conflicts with ON CONFLICT
    clause to update existing records.
    """
    col = mongo[COL_ACS]
    cur = pg.cursor()

    sql = """
    INSERT INTO acs_zip_population
    (year, zip_code, zip_city, state_2, population, data_class,
     data_field_display_name, data_field, data_stream)
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)
    ON CONFLICT (year, zip_code) DO UPDATE SET
      zip_city = EXCLUDED.zip_city,
      state_2 = EXCLUDED.state_2,
      population = EXCLUDED.population,
      data_class = EXCLUDED.data_class,
      data_field_display_name = EXCLUDED.data_field_display_name,
      data_field = EXCLUDED.data_field,
      data_stream = EXCLUDED.data_stream;
    """

    batch = []
    for d in col.find({}, {"_id": 0}):
        batch.append((
            d.get("year"),
            d.get("zip_code"),
            d.get("zip_city"),
            d.get("state_2"),
            d.get("population"),
            d.get("data_class"),
            d.get("data_field_display_name"),
            d.get("data_field"),
            d.get("data_stream"),
        ))

        if len(batch) >= 5000:
            execute_batch(cur, sql, batch)
            batch.clear()

    if batch:
        execute_batch(cur, sql, batch)

    cur.close()
    print("Population data migrated")


In [25]:
#execute
if __name__ == "__main__":
    try:
        migrate_acs()
    except Exception as e:
        print(f"Migration failed: {type(e).__name__}: {e}")
    finally:
        if 'pg' in locals():
            pg.close()
            print("PostgreSQL connection closed.")


Population data migrated
PostgreSQL connection closed.
